# 🏁 Pandas Final Project — Expense Analysis
Start with this dataset:

In [58]:
import pandas as pd

df = pd.DataFrame({
    "date": pd.to_datetime([
        "2026-01-02", "2026-01-03", "2026-01-03",
        "2026-01-05", "2026-01-06", "2026-01-08",
        "2026-01-10", "2026-01-12", "2026-01-13",
        "2026-01-15", "2026-01-17", "2026-01-18"
    ]),
    "category": [
        "Food", "Transport", "Food",
        "Shopping", "Bills", "Food",
        "Transport", "Food", "Entertainment",
        "Shopping", "Bills", "Food"
    ],
    "amount": [
        250, 120, 300,
        800, 1000, 180,
        150, 350, 450,
        600, 900, 220
    ]
})

df

,date,category,amount
0,2026-01-02,Food,250
1,2026-01-03,Transport,120
2,2026-01-03,Food,300
3,2026-01-05,Shopping,800
4,2026-01-06,Bills,1000
5,2026-01-08,Food,180
6,2026-01-10,Transport,150
7,2026-01-12,Food,350
8,2026-01-13,Entertainment,450
9,2026-01-15,Shopping,600


## Part 1 — Basic Statistics

Your first task is to calculate:

1. Total spending
2. Average spending
3. Highest expense
4. Lowest expense

In [59]:
total_spending = df["amount"].sum()
avg_spending = df["amount"].mean()
highest_spending = df["amount"].max()
lowest_spending = df["amount"].min()

print(f"Total spending = {total_spending}")
print(f"Average spending = {avg_spending}")
print(f"Highest spending = {highest_spending}")
print(f"Lowest_spending = {lowest_spending}")

Total spending = 5320
Average spending = 443.3333333333333
Highest spending = 1000
Lowest_spending = 120


## Part 2 — Category Analysis

### A. Total spending per category
```text
Create a DataFrame called:

category_summary

that looks roughly like:

category          amount
Bills             ...
Entertainment     ...
Food              ...
Shopping          ...
Transport         ...
```

In [60]:
category_summary = df.groupby("category")["amount"].sum().reset_index()

category_summary

,category,amount
0,Bills,1900
1,Entertainment,450
2,Food,1300
3,Shopping,1400
4,Transport,270


### B. Highest-spending category
```text
Create:

highest_category

containing the category with the largest total spending.
```

In [61]:
highest_category = category_summary.sort_values(
    by = "amount",
    ascending = False
).iloc[0]

highest_category

category    Bills
amount       1900
Name: 0, dtype: object

### C. Average spending per category
```text
Create:

category_average

containing the average expense for each category.
```

In [62]:
category_average = df.groupby("category")["amount"].mean().reset_index()
category_average

,category,amount
0,Bills,950.0
1,Entertainment,450.0
2,Food,260.0
3,Shopping,700.0
4,Transport,135.0


## Part 3 — Time Analysis 📅

Now we're going to use the time-series skills you just learned.

We want to determine:

1. Total spending per week
2. Which week had the highest spending
3. Percentage change between weeks

In [63]:
# Total spending per week

df = df.set_index("date")

weekly_summary = df["amount"].resample("W").sum().reset_index()

weekly_summary

,date,amount
0,2026-01-04,670
1,2026-01-11,2130
2,2026-01-18,2520


In [64]:
# Which week had the highest spending

highest_week = weekly_summary.sort_values(
    by = "amount",
    ascending=False
).iloc[0]

highest_week

date      2026-01-18 00:00:00
amount                   2520
Name: 2, dtype: object

In [65]:
# Percentage change between weeks

weekly_percentage_change = (
    (df["amount"].resample("W").sum().pct_change() * 100)
)
weekly_percentage_change

date
2026-01-04           NaN
2026-01-11    217.910448
2026-01-18     18.309859
Freq: W-SUN, Name: amount, dtype: float64

## Part 4 — Spending Classification
```text
Now we're moving into the transformation part of the project.

Create a new column called:

spending_level

with these rules:
```
| Amount | Level |
| :--- | :--- |
| < 200 | "Low" |
| 200–500 | "Medium" |
| > 500 | "High" |

In [66]:
def level(amount):
    if amount < 200:
        return "Low"
    elif amount > 500:
        return "High"
    else:
        return "Medium"

df["spending_level"] = df["amount"].apply(level)
df    

,category,amount,spending_level
date,,,
2026-01-02,Food,250,Medium
2026-01-03,Transport,120,Low
2026-01-03,Food,300,Medium
2026-01-05,Shopping,800,High
2026-01-06,Bills,1000,High
2026-01-08,Food,180,Low
2026-01-10,Transport,150,Low
2026-01-12,Food,350,Medium
2026-01-13,Entertainment,450,Medium


## Part 5 — Category Average with transform()
```text
Now we'll use one of the more important Pandas techniques you've learned.

We want a new column:

category_average

For every row, it should contain the average spending for that row's category.

For example, if Food has:

250
300
180
350
220

then every Food row should have the same category average.
```

In [67]:
df["category_average"] = (
    df.groupby("category")["amount"]
    .transform("mean")
)

df

,category,amount,spending_level,category_average
date,,,,
2026-01-02,Food,250,Medium,260.0
2026-01-03,Transport,120,Low,135.0
2026-01-03,Food,300,Medium,260.0
2026-01-05,Shopping,800,High,700.0
2026-01-06,Bills,1000,High,950.0
2026-01-08,Food,180,Low,260.0
2026-01-10,Transport,150,Low,135.0
2026-01-12,Food,350,Medium,260.0
2026-01-13,Entertainment,450,Medium,450.0


## Part 6 — Difference from Category Average
```text
Now let's make the analysis more useful.

Create a column:

difference_from_category_average

It should answer:

How much more or less did this particular expense cost compared with the average expense for its category?

For example:

amount = 300
category_average = 260

difference = 40

And:

amount = 150
category_average = 260

difference = -110
```

In [68]:
df["difference_from_category_average"] = df["amount"] - df["category_average"]

df

,category,amount,spending_level,category_average,difference_from_category_average
date,,,,,
2026-01-02,Food,250,Medium,260.0,-10.0
2026-01-03,Transport,120,Low,135.0,-15.0
2026-01-03,Food,300,Medium,260.0,40.0
2026-01-05,Shopping,800,High,700.0,100.0
2026-01-06,Bills,1000,High,950.0,50.0
2026-01-08,Food,180,Low,260.0,-80.0
2026-01-10,Transport,150,Low,135.0,15.0
2026-01-12,Food,350,Medium,260.0,90.0
2026-01-13,Entertainment,450,Medium,450.0,0.0


## Part 7 — Final Report
```text
Now let's bring the project together.

We want to create a category report containing:

Total spending
Average spending
Number of expenses
Highest expense
Lowest expense

for each category.

The final table should conceptually look like:

category        total    average    count    max    min
Bills           ...       ...        ...     ...    ...
Entertainment   ...       ...        ...     ...    ...
Food            ...       ...        ...     ...    ...
Shopping        ...       ...        ...     ...    ...
Transport       ...       ...        ...     ...    ...
```

In [69]:
category_report = df.groupby("category")["amount"].agg(
    ["sum", "mean", "count", "max", "min"]
).reset_index()

category_report

,category,sum,mean,count,max,min
0,Bills,1900,950.0,2,1000,900
1,Entertainment,450,450.0,1,450,450
2,Food,1300,260.0,5,350,180
3,Shopping,1400,700.0,2,800,600
4,Transport,270,135.0,2,150,120


In [70]:
category_report = category_report.rename(
    columns = {
        "sum": "total_spending",
        "mean": "average_spending",
        "count": "number_of_expenses",
        "max": "highest_expens",
        "min": "lowest_expense"
    }
)

category_report

,category,total_spending,average_spending,number_of_expenses,highest_expens,lowest_expense
0,Bills,1900,950.0,2,1000,900
1,Entertainment,450,450.0,1,450,450
2,Food,1300,260.0,5,350,180
3,Shopping,1400,700.0,2,800,600
4,Transport,270,135.0,2,150,120


## Part 6 — Weekly Report
```text
We already calculated weekly totals and percentage changes separately. Now let's make one clean DataFrame containing:

week
total_spending
percentage_change
```

In [71]:
weekly_summary["percentage_change"] = (
    weekly_summary["amount"].pct_change() * 100
)

weekly_summary

,date,amount,percentage_change
0,2026-01-04,670,NaN
1,2026-01-11,2130,217.910448
2,2026-01-18,2520,18.309859


In [72]:
weekly_summary = weekly_summary.rename(
    columns = {
        "date": "week",
        "amount": "total_spending"
    }
)

weekly_summary

,week,total_spending,percentage_change
0,2026-01-04,670,NaN
1,2026-01-11,2130,217.910448
2,2026-01-18,2520,18.309859


# 🏁 Final step — Export the project

Once you've renamed the columns, save your main transaction DataFrame as:
```python
df.to_csv("final_expense_analysis.csv", index=False)
```
And save your reports separately:
```python
category_report.to_csv("category_report.csv", index=False)

weekly_summary.to_csv("weekly_report.csv", index=False)
```
That gives you three useful outputs:

* final_expense_analysis.csv
* category_report.csv
* weekly_report.csv

### Your three files should represent:

| File | Contains |
| :--- | :--- |
| final_expense_analysis.csv | Individual expenses + calculated columns | 
| category_report.csv | Category-level summary |
| weekly_report.csv | Weekly spending + percentage changes |

One more thing: because you did df = df.set_index("date") earlier, check whether your date is currently the index. If it is, index=False would exclude the date from the CSV. In that case, before exporting:

In [73]:
df = df.reset_index()

In [74]:
df.to_csv("final_expense_analysis.csv", index=False)

In [75]:
category_report.to_csv("category_report.csv", index=False)
weekly_summary.to_csv("weekly_report.csv", index=False)

In [76]:
print("End of project")

End of project
